<a href="https://colab.research.google.com/github/roserocarlos/StatAI-Basics/blob/main/Ejercicio4/Cultivating_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cultivating Machine Learning
### Predicción del estado de salud de un cultivo a partir de sensores IoT reales

Dataset: **Plant_health_data.csv** (Kaggle — [gowthamduggirala/plant-health-data](https://www.kaggle.com/datasets/gowthamduggirala/plant-health-data)), 1200 registros reales con 14 columnas de sensores agrícolas.

Objetivo: predecir `Plant_Health_Status` (Healthy / Moderate Stress / High Stress) a partir de 10 sensores numéricos. El notebook está organizado en 8 sesiones incrementales — el código de cada una continúa el de la anterior, y la Sesión 8 usa todo el pipeline acumulado. Dentro de cada sesión, el código de trabajo está partido en bloques cortos, uno por tarea, cada uno con una línea de texto explicando qué hace antes de la celda. Al final de cada sesión hay una celda corta de **visualización de apoyo**, separada del código de trabajo, solo para ver de un vistazo lo más representativo de esa sesión.

## Paso 0 — Cargar el dataset

Descarga `Plant_health_data.csv` desde [Kaggle](https://www.kaggle.com/datasets/gowthamduggirala/plant-health-data) a tu computador y súbelo cuando la celda lo pida.

In [ ]:
from google.colab import files
import os

print("Sube tu archivo CSV de Kaggle (Plant_health_data.csv):")
uploaded = files.upload()

os.makedirs('./data', exist_ok=True)
for filename in uploaded.keys():
    os.rename(filename, os.path.join('./data', filename))
    print(f"Archivo '{filename}' guardado en './data/' exitosamente.")

!ls ./data

## Sesión 1 — Del dato al DataFrame

El dato rectangular (Data Frame): fila = registro, columna = feature/predictor. Cargamos el CSV y separamos los 10 sensores numéricos del target categórico `Plant_Health_Status`.

La búsqueda del archivo con `glob` es insensible a mayúsculas/minúsculas: Kaggle a veces exporta el CSV como `plant_health_data.csv` (todo en minúscula) en vez de `Plant_health_data.csv`.

**Bloque 1 — Detectar y cargar el archivo:**

In [ ]:
import pandas as pd
import numpy as np
import glob

candidatos = glob.glob("./data/*lant_health_data*.csv") + glob.glob("./data/*Plant_health_data*.csv")
RUTA_CSV = sorted(set(candidatos))[0]
print(f"Archivo detectado: {RUTA_CSV}")

df = pd.read_csv(RUTA_CSV, parse_dates=["Timestamp"])

**Bloque 2 — Ver el tamaño y los tipos de dato:**

In [ ]:
print("--- SESION 1: Dimensiones del dataset ---")
print(f"Registros (filas): {df.shape[0]} | Caracteristicas (columnas): {df.shape[1]}\n")
print("--- Tipos de datos por sensor ---")
print(df.dtypes)

**Bloque 3 — Definir los sensores predictores y el target (los vas a reutilizar en todas las sesiones siguientes):**

In [ ]:
SENSORES_NUM = ["Soil_Moisture", "Ambient_Temperature", "Soil_Temperature",
                "Humidity", "Light_Intensity", "Soil_pH",
                "Nitrogen_Level", "Phosphorus_Level", "Potassium_Level",
                "Electrochemical_Signal"]
TARGET = "Plant_Health_Status"

**Visualización de apoyo — distribución del target:**

In [ ]:
import matplotlib.pyplot as plt

df[TARGET].value_counts().plot(kind="bar", color="teal", edgecolor="black")
plt.title("Registros por clase de Plant_Health_Status")
plt.ylabel("Cantidad de registros")
plt.tight_layout()
plt.show()

**Ejercicio:** confirma cuántos registros y columnas tiene tu copia del dataset, y verifica que `SENSORES_NUM` coincide exactamente con los nombres de columnas que ves en `df.dtypes`.

*Pista:* `df.shape` te da (filas, columnas); compara `SENSORES_NUM` contra `list(df.columns)` imprimiendo ambos, o revisando si cada nombre de `SENSORES_NUM` aparece en `df.columns`.

## Sesión 2 — Estadística descriptiva, moda y limpieza según el origen del sensor

Añadimos la **moda** a la media y la mediana. Sobre los nulos: los contamos primero — no se asume que "los sensores IoT siempre fallan"; se revisa. En la descarga real de Kaggle este dataset **no trae valores faltantes**.

Aun así, dejamos una limpieza defensiva por si tu copia del CSV sí trae huecos — y la estrategia depende del **origen físico** de cada sensor, no solo de la estadística: `Soil_Moisture`, `Light_Intensity`, `Ambient_Temperature` y `Humidity` no oscilan alrededor de un centro, siguen un proceso direccional (riego → secado, ciclo diurno). Verificamos que el dataset trae lecturas cada 6 horas sin huecos por planta (`Plant_ID`) — 120 lecturas por planta, siempre con un salto de exactamente 6 horas — así que ahí un valor faltante se interpola dentro de la serie temporal de esa misma planta, en vez de rellenarlo con un promedio global que ignora ese suceso físico. `Soil_pH` y los nutrientes (N, P, K) cambian más lento y sin ese patrón cíclico marcado, así que la mediana global sigue siendo un sustituto razonable para ellos.

**Bloque 1 — Estadísticas descriptivas con moda:**

In [ ]:
print("--- SESION 2: Estadisticas Descriptivas Iniciales ---")
estadisticas = df[SENSORES_NUM].describe().T[["mean", "50%", "std", "min", "max"]]
estadisticas["moda"] = df[SENSORES_NUM].mode().iloc[0]
print(estadisticas)

**Bloque 2 — Contar los nulos reales antes de limpiar nada:**

In [ ]:
nulos_antes = df[SENSORES_NUM].isnull().sum()
print("Valores nulos detectados por sensor:")
print(nulos_antes)
print(f"\nTotal de nulos en el dataset: {nulos_antes.sum()}")

**Bloque 3 — Limpiar según el origen físico del sensor:** interpolación dentro de la serie de cada planta para los sensores "direccionales" (`VARIABLES_TEMPORALES`), mediana global para los que cambian lento (`VARIABLES_LENTAS`):

In [ ]:
VARIABLES_TEMPORALES = ["Soil_Moisture", "Light_Intensity", "Ambient_Temperature", "Humidity"]
VARIABLES_LENTAS = ["Soil_Temperature", "Soil_pH", "Nitrogen_Level", "Phosphorus_Level",
                     "Potassium_Level", "Electrochemical_Signal"]

df = df.sort_values(["Plant_ID", "Timestamp"])
for variable in VARIABLES_TEMPORALES:
    df[variable] = df.groupby("Plant_ID")[variable].transform(
        lambda serie: serie.interpolate(method="linear", limit_direction="both")
    )
for variable in VARIABLES_LENTAS:
    df[variable] = df[variable].fillna(df[variable].median())

df = df.dropna(subset=[TARGET])

**Bloque 4 — Verificar que la limpieza funcionó (o que no hacía falta):**

In [ ]:
nulos_despues = df[SENSORES_NUM].isnull().sum()
print("Valores nulos despues de la limpieza (0 en este dataset: no habia huecos que llenar):")
print(nulos_despues)

**Visualización de apoyo — nulos antes vs. después de imputar:**

In [ ]:
pd.DataFrame({"antes": nulos_antes, "despues": nulos_despues}).plot(
    kind="bar", figsize=(8, 3), color=["#C9552E", "#2E7D32"]
)
plt.title("Valores nulos por sensor: antes vs. despues de la imputacion")
plt.ylabel("Cantidad de nulos")
plt.tight_layout()
plt.show()

**Ejercicio:** calcula la media y la mediana de **cada uno** de los 10 sensores en `SENSORES_NUM`, y la diferencia relativa entre ambas: `abs(media - mediana) / mediana`. ¿Cuál sensor tiene la mayor diferencia? Con esos números, ¿dirías que este dataset tiene sensores con outliers fuertes, o distribuciones bastante simétricas? Justifica con las cifras que obtuviste (no lo respondas "a ojo").

*Pista:* recorre `SENSORES_NUM` con un `for sensor in SENSORES_NUM:`, y dentro calcula `df[sensor].mean()` y `df[sensor].median()`; guarda o imprime la diferencia relativa de cada uno para poder compararlas al final.

## Sesión 3 — Análisis Exploratorio de Datos (EDA)

Filosofía de Tukey: "el modelo siempre debe seguir a los datos". Empezamos con la correlación de Pearson entre los 10 sensores (relación sensor-contra-sensor). Pero lo que de verdad importa para este proyecto es si un sensor distingue las 3 clases de `Plant_Health_Status` — eso es sensor-contra-**objetivo**, y como el objetivo es categórico, Pearson no aplica directamente. Para verlo *antes* de entrenar cualquier modelo, usamos boxplots por clase y el ANOVA F-test.

**Bloque 1 — Correlación entre sensores (Pearson):**

In [ ]:
print("--- SESION 3: Correlacion entre sensores ---")

matriz_corr = df[SENSORES_NUM].corr()
print("Matriz de correlacion de Pearson (sensores numericos):")
print(matriz_corr.round(2))

**Bloque 2 — ¿Qué sensores distinguen mejor las 3 clases de salud?** Un boxplot por clase deja ver a simple vista si el sensor separa a las plantas sanas de las estresadas:

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, sensor in zip(axes.flatten(), SENSORES_NUM):
    df.boxplot(column=sensor, by=TARGET, ax=ax)
    ax.set_title(sensor, fontsize=9)
    ax.set_xlabel("")
plt.suptitle("Cada sensor, agrupado por clase de Plant_Health_Status")
plt.subplots_adjust(hspace=0.4, wspace=0.4)
plt.show()

**Bloque 3 — Confirmar la impresión visual con números:** el ANOVA F-test mide qué tanto separa cada sensor a las clases (F alto y p-valor muy por debajo de 0.05 = separación real, no ruido):

In [ ]:
from sklearn.feature_selection import f_classif

F_valores, p_valores = f_classif(df[SENSORES_NUM], df[TARGET])
anova = pd.DataFrame({"sensor": SENSORES_NUM, "F": F_valores, "p_valor": p_valores})
anova = anova.sort_values("F", ascending=False)
print("ANOVA F-test: que tanto separa cada sensor las 3 clases de salud")
print(anova.to_string(index=False))

**Ejercicio:** con los boxplots y la tabla de ANOVA, ¿qué 2 o 3 sensores separan mejor las 3 clases de salud? Anótalos — en la Sesión 7 vas a comparar esta lista contra el ranking de importancia del Random Forest, para ver si el modelo "descubre" lo mismo que ya viste aquí, solo con estadística descriptiva.

*Pista:* en la tabla `anova`, ya está ordenada de mayor a menor `F`; los primeros de la lista son los candidatos.

**Visualización de apoyo — distribución de Light_Intensity:**

In [ ]:
df["Light_Intensity"].hist(bins=25, color="teal", edgecolor="black", figsize=(6, 3))
plt.title("Distribucion de Intensidad Luminica (lux)")
plt.xlabel("Light_Intensity"); plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

**Visualización de apoyo — ¿hay un patrón según la hora de lectura?**

Las lecturas se toman cada 6 horas, siempre a las mismas 4 horas del día: 4:00, 10:00, 16:00 y 22:00 (no hay lecturas a cualquier hora del día). Por eso, un gráfico de dispersión coloreado "día vs. noche" no se ve como una nube continua, sino como columnas apiladas en esas 4 horas exactas. Un gráfico de barras con el **promedio** por hora es más claro para esta pregunta.

In [ ]:
df["hora_lectura"] = df["Timestamp"].dt.hour
promedio_por_hora = df.groupby("hora_lectura")["Light_Intensity"].mean()

print("Promedio de Light_Intensity por hora de lectura:")
print(promedio_por_hora)

promedio_por_hora.plot(kind="bar", color="#C9A227", edgecolor="black", figsize=(6, 3))
plt.title("Light_Intensity promedio segun hora de lectura")
plt.xlabel("Hora del dia"); plt.ylabel("Light_Intensity promedio")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Ejercicio:** con el gráfico anterior, ¿ves un ciclo día/noche marcado en `Light_Intensity`? Compara qué tan distintos son los 4 promedios entre sí frente al rango total de la variable (aprox. 300–900 lux). ¿Qué podría explicar que un cultivo no muestre ese ciclo (piensa en invernadero, luz artificial o control ambiental)?

*Pista:* la diferencia entre el promedio más alto y el más bajo de `promedio_por_hora` la obtienes con `promedio_por_hora.max() - promedio_por_hora.min()`.

**Ejercicio:** identifica el par de sensores con la correlación más alta (en valor absoluto, sin contar la diagonal) y el par con la correlación más cercana a cero, usando `matriz_corr`. Interpreta ambos casos en el contexto agronómico del cultivo.

*Pista:* `matriz_corr.abs()` te da los valores absolutos; recorre la matriz con dos ciclos `for` anidados y recuerda excluir la diagonal (donde cada sensor se compara consigo mismo y la correlación siempre da 1.0).

## Sesión 4 — Relaciones y transformación de escala de potencias (Tukey)

`log10(x + 1)` es una herramienta clásica para linealizar relaciones cuando una variable tiene sesgo fuerte (cola larga). En **este dataset en particular**, al calcular `Light_Intensity.skew()` el valor sale cercano a 0 (datos de un cultivo con condiciones bastante controladas, sin outliers extremos) — así que el transform casi no va a cambiar la correlación, y eso también es un hallazgo válido: no toda variable numérica necesita transformarse. Aun así aplicamos la técnica sobre `Light_Intensity` para que quede clara la mecánica (y el porqué del `+1`), porque sí es indispensable cuando trabajes con datos de campo reales que tengan sesgo fuerte.

**Bloque 1 — Medir el sesgo (skew) de los 10 sensores:**

In [ ]:
print("--- SESION 4: Sesgo (skew) de los sensores ---")
print(df[SENSORES_NUM].skew().sort_values(key=abs, ascending=False))

**Bloque 2 — Aplicar `log10(x+1)` a `Light_Intensity` y comparar la correlación con `Chlorophyll_Content` antes/después:**

In [ ]:
df["Light_Intensity_log"] = np.log10(df["Light_Intensity"] + 1)

corr_original = df["Light_Intensity"].corr(df["Chlorophyll_Content"])
corr_transf = df["Light_Intensity_log"].corr(df["Chlorophyll_Content"])
print(f"Correlacion original Light vs Clorofila: {corr_original:.4f}")
print(f"Correlacion log(Light) vs Clorofila:      {corr_transf:.4f}")

Como la correlación casi no cambió (era lo esperado, dado el sesgo cercano a 0 que calculaste en el Bloque 1), en la Sesión 5 seguimos usando `Light_Intensity` **sin transformar** como predictor: dejamos que los datos decidan si vale la pena una transformación, en vez de forzarla en el pipeline final solo porque la técnica existe.

**Visualización de apoyo — Soil_Moisture vs. Chlorophyll_Content:**

In [ ]:
plt.figure(figsize=(6, 3))
plt.scatter(df["Soil_Moisture"], df["Chlorophyll_Content"], alpha=0.5, color="tomato")
plt.title("Humedad de Suelo vs Contenido de Clorofila")
plt.xlabel("Soil_Moisture (%)"); plt.ylabel("Chlorophyll_Content")
plt.tight_layout()
plt.show()

**Ejercicio:** ¿por qué se usa `log10(x + 1)` y no `log10(x)` directamente? Mirando el resultado del Bloque 1, ¿cuál sensor tiene el sesgo más alto (en valor absoluto)? Aplica `np.log10(df[sensor] + 1)` a ese sensor y compara su correlación con `Chlorophyll_Content` antes y después. ¿Cambió mucho? ¿Por qué sí o por qué no, dado lo que viste en la introducción de esta sesión?

*Pista:* ya tienes `df[SENSORES_NUM].skew()` calculado en el Bloque 1 — no hace falta recalcularlo.

## Sesión 5 — Partición y estandarización (Z-score)

Estandarizamos con `StandardScaler` ajustado **solo** en entrenamiento (evita fuga de datos) y particionamos 80/20 de forma estratificada.

**Bloque 1 — Elegir los predictores y codificar el target:** usamos los 10 sensores de `SENSORES_NUM` sin transformar (la Sesión 4 mostró que el log no aporta aquí):

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

predictores = list(SENSORES_NUM)

X = df[predictores]
le = LabelEncoder()
y = le.fit_transform(df[TARGET])   # Healthy / Moderate Stress / High Stress -> 0,1,2
print(f"Clases codificadas: {dict(zip(le.classes_, le.transform(le.classes_)))}")

**Bloque 2 — Particionar 80/20 de forma estratificada** (para no perder la clase minoritaria "High Stress" en el set de prueba):

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Entrenamiento: {X_train.shape} | Prueba: {X_test.shape}")

**Bloque 3 — Estandarizar con Z-score** (el `scaler` se ajusta solo con entrenamiento, para no filtrar información del set de prueba):

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Visualización de apoyo — tamaño de los conjuntos:**

In [ ]:
plt.bar(["Entrenamiento", "Prueba"], [len(y_train), len(y_test)], color=["#2E7D32", "#C9A227"])
plt.title("Registros por conjunto")
plt.ylabel("Cantidad de registros")
plt.tight_layout()
plt.show()

**Ejercicio:** quita `stratify=y` de `train_test_split` (dejando el resto de argumentos igual), vuelve a correr la celda y compara cuántos registros de la clase "High Stress" quedan en `y_test`. ¿Por qué importa esto para una clase minoritaria?

*Pista:* usa `pd.Series(y_test).value_counts()` antes y después del cambio para contar cuántos registros hay de cada clase codificada (revisa `dict(zip(le.classes_, le.transform(le.classes_)))` del Bloque 1 para saber cuál número es "High Stress").

## Sesión 6 — Modelo lineal vs. Árbol (CART)

Regresión Logística (paramétrica, interpretable) vs. Árbol de Decisión (no paramétrico, captura interacciones no lineales). Comparamos con Accuracy y F1-macro.

**Bloque 1 — Regresión Logística:**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)
pred_log = log_model.predict(X_test_scaled)

**Bloque 2 — Árbol CART:**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train_scaled, y_train)
pred_tree = tree_model.predict(X_test_scaled)

**Bloque 3 — Comparar ambos modelos:**

In [ ]:
acc_log = accuracy_score(y_test, pred_log); f1_log = f1_score(y_test, pred_log, average="macro")
acc_tree = accuracy_score(y_test, pred_tree); f1_tree = f1_score(y_test, pred_tree, average="macro")

print(f"Regresion Logistica -> Accuracy: {acc_log:.4f} | F1-macro: {f1_log:.4f}")
print(f"Arbol CART           -> Accuracy: {acc_tree:.4f} | F1-macro: {f1_tree:.4f}")

**Visualización de apoyo — F1-macro por modelo:**

In [ ]:
plt.bar(["Regresion Logistica", "Arbol CART"], [f1_log, f1_tree], color=["#C9552E", "#2E7D32"])
plt.title("F1-macro: Logistica vs CART")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

**Ejercicio:** cambia `max_depth=4` a `max_depth=2` y luego a `max_depth=10` dentro de `DecisionTreeClassifier(...)`, sin tocar nada más de la celda. ¿Qué pasa con `f1_tree` en cada caso? Relaciónalo con el concepto de sobreajuste (overfitting).

*Pista:* solo edita el argumento `max_depth` y vuelve a correr el Bloque 2 y el Bloque 3 cada vez, para que `pred_tree` y `f1_tree` se recalculen con el nuevo árbol.

## Sesión 7 — Ensambles: Random Forest y XGBoost

Bagging (árboles en paralelo sobre muestras bootstrap) vs. Boosting (árboles secuenciales que corrigen el error residual). Validamos con 5-fold cross-validation y extraemos la importancia de variables — la vamos a comparar con lo que ya habías anotado en el ejercicio de boxplots/ANOVA de la Sesión 3.

**Bloque 1 — Random Forest** (instala `xgboost` de una vez, lo usamos en el siguiente bloque):

In [ ]:
!pip install -q xgboost
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train_scaled, y_train)
pred_rf = rf_model.predict(X_test_scaled)

**Bloque 2 — XGBoost:**

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3,
                           random_state=42, eval_metric="mlogloss")
xgb_model.fit(X_train_scaled, y_train)
pred_xgb = xgb_model.predict(X_test_scaled)

**Bloque 3 — Validación cruzada de 5 folds sobre Random Forest** (para confirmar que el resultado no depende de un solo split de datos):

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring="f1_macro")
print(f"Random Forest (F1-macro promedio, 5-Fold CV): {cv_scores.mean():.4f}")

**Bloque 4 — Importancia de variables:**

In [ ]:
importancias = rf_model.feature_importances_
ranking = sorted(zip(predictores, importancias), key=lambda x: x[1], reverse=True)
print("Ranking de importancia de variables (Random Forest):")
for var, imp in ranking:
    print(f" -> {var:<24}: {imp*100:5.2f}%")

**Visualización de apoyo — importancia de variables:**

In [ ]:
nombres, valores = zip(*ranking)
plt.barh(nombres[::-1], valores[::-1], color="#2E7D32")
plt.title("Importancia de variables (Random Forest)")
plt.tight_layout()
plt.show()

**Ejercicio:** anota los 3 sensores con mayor importancia en `ranking` para tu propia corrida (el orden puede variar un poco entre ejecuciones, ya que Random Forest usa aleatoriedad). ¿Coinciden con los sensores que ya habías anotado en el ejercicio de boxplots/ANOVA de la Sesión 3? Propón una explicación agronómica de por qué esos sensores dominan la predicción.

*Pista:* `ranking` ya viene ordenado de mayor a menor importancia; `ranking[:3]` te da directamente los tres primeros pares (sensor, importancia).

## Sesión 8 — Pipeline consolidado y evaluación final

Comparamos los 4 modelos, elegimos el mejor por F1-macro y generamos matriz de confusión y `classification_report` del ganador.

**Bloque 1 — Métricas de Random Forest y XGBoost, y tabla comparativa de los 4 modelos:**

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

acc_rf = accuracy_score(y_test, pred_rf); f1_rf = f1_score(y_test, pred_rf, average="macro")
acc_xgb = accuracy_score(y_test, pred_xgb); f1_xgb = f1_score(y_test, pred_xgb, average="macro")

print("=======================================================")
print("        TABLA DE RENDIMIENTO FINAL DE MODELOS          ")
print("=======================================================")
print(f" 1. Regresion Logistica  -> Accuracy: {acc_log:.4f} | F1-macro: {f1_log:.4f}")
print(f" 2. Arbol CART           -> Accuracy: {acc_tree:.4f} | F1-macro: {f1_tree:.4f}")
print(f" 3. Random Forest        -> Accuracy: {acc_rf:.4f} | F1-macro: {f1_rf:.4f}")
print(f" 4. XGBoost              -> Accuracy: {acc_xgb:.4f} | F1-macro: {f1_xgb:.4f}")
print("=======================================================")

**Bloque 2 — Elegir el mejor modelo por F1-macro:**

In [ ]:
resultados = {"Regresion Logistica": f1_log, "Arbol CART": f1_tree,
              "Random Forest": f1_rf, "XGBoost": f1_xgb}
mejor_modelo = max(resultados, key=resultados.get)
print(f"Modelo recomendado (mayor F1-macro): {mejor_modelo}")

**Bloque 3 — Detalle del mejor modelo: matriz de confusión y reporte de clasificación:**

In [ ]:
print("Matriz de confusion - mejor modelo (Random Forest):")
print(confusion_matrix(y_test, pred_rf))
print("\nReporte de clasificacion (Random Forest):")
print(classification_report(y_test, pred_rf, target_names=le.classes_))

print("\nEl pipeline incremental de 8 sesiones se ejecuto correctamente de principio a fin.")

**Visualización de apoyo — comparativa final de los 4 modelos:**

In [ ]:
plt.bar(list(resultados.keys()), list(resultados.values()), color=["#C9552E", "#2E7D32", "#7FB88A", "#C9A227"])
plt.title("F1-macro por modelo")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

**Ejercicio:** el Árbol CART y el Random Forest llegan a 100% de accuracy en el set de prueba. Para un problema real de campo, eso no debería ser motivo de celebración automática, sino de sospecha: ningún sensor agrícola mide sin ruido, y una separación perfecta suele indicar que el dataset se generó con reglas o umbrales simples en vez de reflejar variabilidad real. ¿Qué evidencia revisarías para descartar (o confirmar) esa sospecha?

*Pista:* usa `df.groupby(TARGET)[sensor].describe()` con `sensor = "Soil_Moisture"` y luego `"Nitrogen_Level"` (las dos variables con más importancia en la Sesión 7). Mira si los rangos (`min`/`max`) de cada clase se solapan mucho entre sí o si están casi perfectamente separados en tercios — eso es una pista de umbrales artificiales.

**Ejercicio final:** con tus propios resultados, escribe 3-4 líneas de conclusión: ¿qué modelo elegirías para desplegar en campo y por qué, considerando tanto el F1-macro como el recall de la clase "High Stress"?

*Pista:* no necesitas código nuevo — el recall por clase ya está impreso en el `classification_report` del Bloque 3, en la fila "High Stress", columna `recall`.